In [1]:
# 6-21-2026

In [24]:
import pandas as pd
import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import spearmanr
import glob
import os

In [17]:
domain_id = "22"

X_train = pd.read_csv(f"train_X/domain_{domain_id}.csv")
y_train = pd.read_csv(f"train_y/domain_{domain_id}.csv")["log_ba"]
X_test = pd.read_csv(f"test_X/domain_{domain_id}.csv")
y_test = pd.read_csv(f"test_y/domain_{domain_id}.csv")["log_ba"]

scaler = joblib.load(f"scalers/domain_{domain_id}.joblib")
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [18]:
# val split only for comparing hyperparam choices, not used in the final loop. rf doesn't need it for fitting itself
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_scaled, y_train, test_size=0.2, random_state=5
)

In [19]:
model = RandomForestRegressor(
    n_estimators=600,
    max_depth=None,
    min_samples_leaf=5,
    max_features="sqrt",
    n_jobs=-1,
    random_state=5
)
model.fit(X_tr, y_tr)

,n_estimators,600
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [20]:
val_pred = model.predict(X_val)
test_pred = model.predict(X_test_scaled)

val_spearman, _ = spearmanr(y_val, val_pred)
test_spearman, _ = spearmanr(y_test, test_pred)
val_r2 = r2_score(y_val, val_pred)
test_r2 = r2_score(y_test, test_pred)

In [21]:
print(f"val spearman: {val_spearman:.4f}, test spearman: {test_spearman:.4f}")
print(f"val r2: {val_r2:.4f}, test r2: {test_r2:.4f}")

val spearman: 0.4584, test spearman: 0.4499
val r2: 0.1314, test r2: 0.1384


In [22]:
RF_PARAMS = {
    "n_estimators": 600,
    "max_depth": None,
    "min_samples_leaf": 5,
    "max_features": "sqrt",
    "n_jobs": -1,
    "random_state": 5
}

In [25]:
domain_files = glob.glob("train_X/domain_*.csv")
domain_ids = sorted(
    int(os.path.basename(f).replace("domain_", "").replace(".csv", ""))
    for f in domain_files
)
print(f"found {len(domain_ids)} domains")

found 34 domains


In [26]:
models = {}
scalers = {}

In [ ]:
# train rf models
for domain_id in domain_ids:
    X_train = pd.read_csv(f"train_X/domain_{domain_id}.csv")
    y_train = pd.read_csv(f"train_y/domain_{domain_id}.csv")["log_ba"] # only series

    scaler = joblib.load(f"scalers/domain_{domain_id}.joblib")
    X_train_scaled = scaler.transform(X_train)

    model = RandomForestRegressor(**RF_PARAMS)
    model.fit(X_train_scaled, y_train)

    models[domain_id] = model
    scalers[domain_id] = scaler

    print(f"domain {domain_id} done")
# takes ~50 min

domain 0 done
domain 1 done
domain 2 done
domain 4 done
domain 5 done
domain 6 done
domain 7 done
domain 8 done
domain 11 done
domain 12 done
domain 13 done
domain 16 done
domain 18 done
domain 19 done
domain 20 done
domain 21 done
domain 22 done
domain 23 done
domain 25 done
domain 26 done
domain 27 done
domain 28 done
domain 29 done
domain 30 done
domain 32 done
domain 33 done
domain 36 done
domain 37 done
domain 38 done
domain 39 done
domain 45 done
domain 46 done
domain 47 done
domain 49 done


In [28]:
test_X_raw = {}
test_y = {}
# load up the testing datasets
for domain_id in domain_ids:
    test_X_raw[domain_id] = pd.read_csv(f"test_X/domain_{domain_id}.csv")
    test_y[domain_id] = pd.read_csv(f"test_y/domain_{domain_id}.csv")["log_ba"]

In [29]:
T_spearman_rf = pd.DataFrame(index=domain_ids, columns=domain_ids, dtype=float)

In [ ]:
for i in domain_ids:
    model_i = models[i]
    scaler_i = scalers[i]

    for j in domain_ids:
        # apply source domain's scaler to target domain's raw test X, not target's own scaler
        X_test_scaled = scaler_i.transform(test_X_raw[j])
        y_true = test_y[j]

        preds = model_i.predict(X_test_scaled)
        T_spearman_rf.loc[i, j], _ = spearmanr(y_true, preds)

    print(f"finished evaluating source domain {i} against all targets")
    # takes ~70 min

finished evaluating source domain 0 against all targets
finished evaluating source domain 1 against all targets
finished evaluating source domain 2 against all targets
finished evaluating source domain 4 against all targets
finished evaluating source domain 5 against all targets
finished evaluating source domain 6 against all targets
finished evaluating source domain 7 against all targets
finished evaluating source domain 8 against all targets
finished evaluating source domain 11 against all targets
finished evaluating source domain 12 against all targets
finished evaluating source domain 13 against all targets
finished evaluating source domain 16 against all targets
finished evaluating source domain 18 against all targets
finished evaluating source domain 19 against all targets
finished evaluating source domain 20 against all targets
finished evaluating source domain 21 against all targets
finished evaluating source domain 22 against all targets
finished evaluating source domain 23 ag

In [31]:
T_spearman_rf

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,0.392855,0.290849,0.128428,0.054446,0.213909,0.087892,0.152805,0.182725,0.175080,0.027583,...,0.226954,0.241617,0.148503,0.272692,0.111680,-0.013762,0.117464,0.163298,0.028433,0.033285
1,0.140050,0.395756,0.049440,0.007159,0.184123,0.066385,0.063603,0.134836,0.104308,0.030660,...,0.173615,0.043663,0.030789,0.193711,0.067833,0.066370,0.113758,0.200665,0.012426,-0.033455
2,0.197478,0.216357,0.368561,0.022614,0.210764,0.146139,0.166881,0.233484,0.156126,0.060366,...,0.199903,0.219109,0.122159,0.253776,0.129618,0.051029,-0.008378,0.260843,0.025048,0.063180
4,0.233598,0.223307,0.128421,0.410325,0.217037,0.183353,0.118123,0.202430,0.169613,0.036186,...,0.181329,0.243651,0.147356,0.284894,0.213711,-0.003993,0.053143,0.305150,0.138506,0.028815
5,0.191233,0.207653,0.094644,0.086692,0.464396,0.146544,0.154414,0.201536,0.261764,0.105262,...,0.199348,0.234613,0.115991,0.294817,0.169369,-0.025139,0.114958,0.298911,0.068162,0.056954
6,0.182795,0.240705,0.039063,0.222890,0.190860,0.317920,0.130435,0.164960,0.125930,0.015462,...,0.200876,0.244970,0.127511,0.255639,0.205514,-0.066257,0.101915,0.321027,0.107055,-0.009512
7,0.107064,0.037322,0.063370,0.066876,0.029070,0.109710,0.328742,0.082349,0.148988,0.050984,...,0.182640,0.177945,0.137945,0.175370,0.210417,0.032822,0.040369,0.155453,0.016488,0.022433
8,0.180067,0.229874,-0.006732,0.020080,0.132165,0.127071,0.191256,0.424810,0.065555,-0.009912,...,0.099800,0.131254,0.100270,0.174319,0.108003,-0.004773,0.049408,0.259646,0.051732,0.001311
11,0.214830,0.226977,0.109047,0.120301,0.238454,0.071556,0.112686,0.206334,0.551493,0.000562,...,0.255409,0.169685,0.102103,0.365516,0.180037,-0.005072,0.146301,0.201576,-0.058867,0.060988
12,0.116046,0.162252,-0.016845,0.108862,0.158614,0.071397,0.081398,0.035134,0.106322,0.502015,...,0.159198,-0.073894,0.073214,0.154261,0.120224,0.083060,0.089615,0.108763,0.069781,-0.070366


In [32]:
T_spearman_rf.to_csv("transfer_matrix_spearman_rf.csv")